# S8.3 — Key Highlights Extraction

Verifies that the `HighlightsService` extracts structured highlights from papers
using the LLM client, and that the API endpoint works correctly.

In [1]:
import sys
sys.path.insert(0, "../..")

from src.schemas.api.analysis import PaperHighlights, HighlightsResponse
import uuid

# Verify PaperHighlights schema
highlights = PaperHighlights(
    novel_contributions=["Introduced self-attention replacing recurrence"],
    important_findings=["BLEU score 28.4 on WMT 2014"],
    practical_implications=["Enables parallel training"],
    limitations=["Quadratic memory complexity"],
    keywords=["transformers", "self-attention", "machine translation"],
)
assert len(highlights.novel_contributions) == 1
assert len(highlights.keywords) == 3
print("\u2713 PaperHighlights schema validates correctly")
print(highlights.model_dump_json(indent=2))

✓ PaperHighlights schema validates correctly
{
  "novel_contributions": [
    "Introduced self-attention replacing recurrence"
  ],
  "important_findings": [
    "BLEU score 28.4 on WMT 2014"
  ],
  "practical_implications": [
    "Enables parallel training"
  ],
  "limitations": [
    "Quadratic memory complexity"
  ],
  "keywords": [
    "transformers",
    "self-attention",
    "machine translation"
  ]
}


In [2]:
# Verify HighlightsResponse schema
paper_id = uuid.UUID("12345678-1234-5678-1234-567812345678")
response = HighlightsResponse(
    paper_id=paper_id,
    highlights=highlights,
    model="gemini-3-flash",
    provider="gemini",
    latency_ms=850.0,
)
assert response.paper_id == paper_id
assert response.model == "gemini-3-flash"
assert response.warning is None
print("\u2713 HighlightsResponse schema validates correctly")
print(response.model_dump_json(indent=2))

✓ HighlightsResponse schema validates correctly
{
  "paper_id": "12345678-1234-5678-1234-567812345678",
  "highlights": {
    "novel_contributions": [
      "Introduced self-attention replacing recurrence"
    ],
    "important_findings": [
      "BLEU score 28.4 on WMT 2014"
    ],
    "practical_implications": [
      "Enables parallel training"
    ],
    "limitations": [
      "Quadratic memory complexity"
    ],
    "keywords": [
      "transformers",
      "self-attention",
      "machine translation"
    ]
  },
  "model": "gemini-3-flash",
  "provider": "gemini",
  "latency_ms": 850.0,
  "warning": null
}


In [3]:
# Verify HighlightsService exists with correct signature
from src.services.analysis.highlights import HighlightsService
import inspect

assert hasattr(HighlightsService, 'extract_highlights')
assert hasattr(HighlightsService, '_prepare_content')
assert hasattr(HighlightsService, '_parse_highlights')

sig = inspect.signature(HighlightsService.extract_highlights)
params = list(sig.parameters.keys())
assert 'paper_id' in params
assert 'force' in params
print("\u2713 HighlightsService has correct methods and signature")
print(f"  extract_highlights params: {params}")

✓ HighlightsService has correct methods and signature
  extract_highlights params: ['self', 'paper_id', 'force']


In [4]:
# Verify content preparation with mock paper
from unittest.mock import AsyncMock, MagicMock

mock_repo = AsyncMock()
mock_llm = AsyncMock()
service = HighlightsService(llm_provider=mock_llm, paper_repo=mock_repo)

paper = MagicMock()
paper.abstract = "We present a novel transformer architecture."
paper.sections = [
    {"title": "Abstract", "content": "We present a novel transformer architecture.", "level": 1},
    {"title": "Results", "content": "Our model achieves 28.4 BLEU.", "level": 1},
    {"title": "Conclusion", "content": "Attention is all you need.", "level": 1},
]
paper.pdf_content = "Full paper text here."

content = service._prepare_content(paper)
assert "Abstract" in content
assert "Results" in content
assert "Conclusion" in content
print("\u2713 Content preparation includes abstract and priority sections")
print(f"  Content length: {len(content.split())} words")

✓ Content preparation includes abstract and priority sections
  Content length: 22 words


In [5]:
# Verify JSON parsing and fallback
import json

valid_json = json.dumps({
    "novel_contributions": ["New method"],
    "important_findings": ["Better results"],
    "practical_implications": ["Useful for X"],
    "limitations": ["Limited scope"],
    "keywords": ["ML", "NLP", "AI"],
})

result, warning = service._parse_highlights(valid_json)
assert warning is None
assert len(result.novel_contributions) == 1
print("\u2713 Valid JSON parsed correctly")

# Test fallback
result2, warning2 = service._parse_highlights("Not valid JSON at all")
assert warning2 is not None
assert len(result2.novel_contributions) >= 1
print("\u2713 Malformed output triggers fallback with warning")

Failed to parse highlights JSON, using fallback: Not valid JSON at all


✓ Valid JSON parsed correctly
✓ Malformed output triggers fallback with warning


In [6]:
# Verify API endpoint is registered
import httpx
from httpx import ASGITransport
from src.main import create_app
from src.services.llm.provider import LLMResponse, UsageMetadata
from src.dependency import get_paper_repository, get_llm_provider
import asyncio

app = create_app()

# Check route exists
routes = [r.path for r in app.routes if hasattr(r, 'path')]
assert any('highlights' in r for r in routes), f"No highlights route found in {routes}"
print("\u2713 POST /api/v1/papers/{{paper_id}}/highlights route registered")

# Test endpoint with mocks
mock_paper = MagicMock()
mock_paper.id = paper_id
mock_paper.abstract = "Test abstract about transformers."
mock_paper.sections = [{"title": "Results", "content": "Good results.", "level": 1}]
mock_paper.pdf_content = "Full text."

mock_repo = AsyncMock()
mock_repo.get_by_id = AsyncMock(return_value=mock_paper)

mock_llm = AsyncMock()
mock_llm.generate = AsyncMock(return_value=LLMResponse(
    text=valid_json, model="test-model", provider="test",
    usage=UsageMetadata(latency_ms=100.0)
))

app.dependency_overrides[get_paper_repository] = lambda: mock_repo
app.dependency_overrides[get_llm_provider] = lambda: mock_llm

async def test_endpoint():
    async with httpx.AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
        resp = await client.post(f"/api/v1/papers/{paper_id}/highlights")
    return resp

resp = await test_endpoint()
assert resp.status_code == 200
data = resp.json()
assert "highlights" in data
assert "novel_contributions" in data["highlights"]
print("\u2713 Endpoint returns 200 with structured highlights")
print(json.dumps(data, indent=2))

app.dependency_overrides.clear()

✓ POST /api/v1/papers/{{paper_id}}/highlights route registered
✓ Endpoint returns 200 with structured highlights
{
  "paper_id": "12345678-1234-5678-1234-567812345678",
  "highlights": {
    "novel_contributions": [
      "New method"
    ],
    "important_findings": [
      "Better results"
    ],
    "practical_implications": [
      "Useful for X"
    ],
    "limitations": [
      "Limited scope"
    ],
    "keywords": [
      "ML",
      "NLP",
      "AI"
    ]
  },
  "model": "test-model",
  "provider": "test",
  "latency_ms": 100.0,
  "warning": null
}
